# 03 - AAA retrieval

Runs 10 clinical AAA queries against the local dense vector index and displays the
retrieved source chunks with their metadata (document, section, page, chunk ID,
similarity score, text).

This notebook retrieves and displays source text only. It does not generate answers.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)

cwd = Path.cwd().resolve()
for candidate in [cwd, cwd / "notebooks", cwd.parent]:
    if (candidate / "clinical_rag.py").exists():
        sys.path.insert(0, str(candidate))
        break
    if (candidate / "notebooks" / "clinical_rag.py").exists():
        sys.path.insert(0, str(candidate / "notebooks"))
        break

import clinical_chunking as cc
import clinical_preprocess as cp
import clinical_rag as cr

PROJECT_ROOT = cp.find_project_root()
index = cr.load_index(PROJECT_ROOT)
# Loaded through cc.load_embedder so the pinned revision applies here too.
model = cc.load_embedder(index["model_name"])

print("Index type:", index["meta"]["index_type"], "| metric:", index["meta"]["metric"])
print("Embedding model:", index["model_name"])
print("Indexed vectors:", index["meta"]["n_vectors"], "x", index["meta"]["embedding_dim"])
print("Clinical queries:", len(cr.CLINICAL_QUERIES))

C:\Users\DELL\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2233.94it/s]

Index type: numpy_cosine | metric: cosine
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Indexed vectors: 1330 x 384
Clinical queries: 10


## Indexed corpus

Only official guideline documents are indexed.

In [2]:
idx_df = pd.DataFrame(index["chunks"])
summary = (
    idx_df.groupby(["document_id", "document_name"])
    .agg(n_chunks=("chunk_id", "size"), pages=("page_number", "nunique"), max_tokens=("token_count", "max"))
    .reset_index()
)
display(summary)
print("total indexed chunks:", len(idx_df))

,document_id,document_name,n_chunks,pages,max_tokens
0,ESVS_2024,Editor's Choice -- European Society for Vascular Surgery (ESVS) 2024 Clinical Practice...,1095,102,224
1,NICE_NG156,Abdominal aortic aneurysm: diagnosis and management,118,50,222
2,SVS_2018,Care of Patients with an Abdominal Aortic Aneurysm,44,37,254
3,USPSTF_2019,Screening for Abdominal Aortic Aneurysm: US Preventive Services Task Force Recommendat...,73,7,222


total indexed chunks: 1330


## Clinical queries and retrieved chunks

Top-5 chunks per query, each shown with document, section, page, chunk ID, similarity
score and the chunk text.

In [3]:
TOP_K = 5
query_runs = []

for q_i, query in enumerate(cr.CLINICAL_QUERIES, start=1):
    hits = cr.retrieve(query, index, model=model, top_k=TOP_K)
    query_runs.append({"query_id": q_i, "query": query, "hits": hits})
    print("=" * 90)
    print(f"Q{q_i}. {query}")
    print("=" * 90)
    for hit in hits:
        print(cr.format_hit(hit, preview_chars=500))
        print("-" * 90)

Q1. What are the recommendations for screening for abdominal aortic aneurysm?
rank 1  score=0.7281
chunk_id: ESVS_2024__p19-19__c0260
document: Editor's Choice -- European Society for Vascular Surgery (ESVS) 2024 Clinical Practice Guidelines on the Management of Abdominal Aorto-Iliac Artery Aneurysms&#x2606; (ESVS_2024)
section: Recommendation 9
page: 19 (pages 19-19)
source_file: ESVS_2024_AAA_Guidelines.pdf
recommendation metadata: recommendation_id=nan, grade=nan, evidence_level=nan
source excerpt: Recommendation 9
Changed
Computed tomography angiography is recommended for treatment planning once the anteroposterior diameter threshold for elective abdominal aortic aneurysm repair has been met on ultrasound, and for the diagnosis of rupture.
Class
Level
References
ToE
I
C
Long et al. (2012),112
Oliver-Williams et al (2019),117
Biancari et al. (2013)122
Recommendation 10
Changed
Aortic diamete
chunk text:
Recommendation 9
Changed
Computed tomography angiography is recommended for trea


Q4. What are the indications for endovascular aneurysm repair?
rank 1  score=0.7132
chunk_id: ESVS_2024__p83-83__c1036
document: Editor's Choice -- European Society for Vascular Surgery (ESVS) 2024 Clinical Practice Guidelines on the Management of Abdominal Aorto-Iliac Artery Aneurysms&#x2606; (ESVS_2024)
section: None
page: 83 (pages 83-83)
source_file: ESVS_2024_AAA_Guidelines.pdf
recommendation metadata: none in this chunk
source excerpt: The aim of surgical treatment of IAAs is to exclude the aneurysm from the circulation to prevent further growth and rupture. Before the advent of endovascular repair in the early
1990s OSR was the mainstay of treatment of IAA. The steady shift towards endovascular techniques since 2000 has been associated with a signiﬁcant decrease in operative morbidity and mortality,1086 and a recent meta-analys
chunk text:
The aim of surgical treatment of IAAs is to exclude the aneurysm from the circulation to prevent further growth and rupture. Before the adve

Q8. What are the recommendations for smoking cessation in patients with AAA?
rank 1  score=0.7535
chunk_id: NICE_NG156__p11-11__c0051
document: Abdominal aortic aneurysm: diagnosis and management (NICE_NG156)
section: Reducing the risk of rupture
page: 11 (pages 11-11)
source_file: abdominal-aortic-aneurysm-diagnosis-and-management-pdf-66141843642565.pdf
recommendation metadata: recommendation_id=1.2.1, grade=nan, evidence_level=nan
source excerpt: For a short explanation of why the committee made these recommendations and how they might affect practice, see the rationale and impact section on providing information to people with a diagnosed AAA.
Full details of the evidence and the committee's discussion are in evidence review K:
effectiveness of endovascular aneurysm repair, open surgical repair and non-surgical management of unruptured ab
chunk text:
For a short explanation of why the committee made these recommendations and how they might affect practice, see the rationale and impa

Q10. What are the differences between open surgical repair and EVAR recommendations?
rank 1  score=0.7949
chunk_id: NICE_NG156__p40-40__c0122
document: Abdominal aortic aneurysm: diagnosis and management (NICE_NG156)
section: Repairing unruptured aneurysms
page: 40 (pages 40-40)
source_file: abdominal-aortic-aneurysm-diagnosis-and-management-pdf-66141843642565.pdf
recommendation metadata: none in this chunk
source excerpt: • has higher net costs and lower net benefits than open surgical repair or
• is substantially above the range NICE normally considers to be a cost-effective use of
NHS resources.
There is a small group of people who have abdominal copathology or other considerations that mean open surgical repair is unsuitable. Examples of copathologies include people who have internal scar-tissue from previous ab
chunk text:
• has higher net costs and lower net benefits than open surgical repair or
• is substantially above the range NICE normally considers to be a cost-effective use

## Top-1 result per query

In [4]:
top1 = pd.DataFrame(
    [
        {
            "query_id": run["query_id"],
            "query": run["query"],
            "document_id": run["hits"][0]["document_id"],
            "section": run["hits"][0]["section"] or "<none>",
            "page": run["hits"][0]["page"],
            "chunk_id": run["hits"][0]["chunk_id"],
            "score": round(run["hits"][0]["similarity_score"], 4),
        }
        for run in query_runs
        if run["hits"]
    ]
)
display(top1)

assert len(query_runs) == len(cr.CLINICAL_QUERIES)
assert all(run["hits"] for run in query_runs), "every query must return chunks"
print(f"\n{len(query_runs)} clinical queries executed; all returned {TOP_K} chunks with metadata.")

,query_id,query,document_id,section,page,chunk_id,score
0,1,What are the recommendations for screening for abdominal aortic aneurysm?,ESVS_2024,Recommendation 9,19,ESVS_2024__p19-19__c0260,0.7281
1,2,What AAA diameter is generally associated with consideration of elective repair?,ESVS_2024,ABDOMINAL AORTIC ANEURYSM,26,ESVS_2024__p26-26__c0351,0.7688
2,3,What surveillance strategy is recommended for small abdominal aortic aneurysms?,ESVS_2024,ABDOMINAL AORTIC ANEURYSM,22,ESVS_2024__p22-22__c0297,0.7383
3,4,What are the indications for endovascular aneurysm repair?,ESVS_2024,<none>,83,ESVS_2024__p83-83__c1036,0.7132
4,5,What are the risk factors associated with abdominal aortic aneurysm?,ESVS_2024,INFORMATION FOR PATIENTS,95,ESVS_2024__p95-95__c1178,0.6718
5,6,What imaging modality is recommended for diagnosis or surveillance of AAA?,ESVS_2024,ABDOMINAL AORTIC ANEURYSM,21,ESVS_2024__p21-21__c0282,0.6702
6,7,What factors influence the risk of AAA rupture?,USPSTF_2019,Summary of Recommendations,2,USPSTF_2019__p2-2__c0016,0.6981
7,8,What are the recommendations for smoking cessation in patients with AAA?,NICE_NG156,Reducing the risk of rupture,11,NICE_NG156__p11-11__c0051,0.7535
8,9,What are the recommendations for women regarding AAA screening?,USPSTF_2019,Treatment,3,USPSTF_2019__p3-3__c0029,0.7064
9,10,What are the differences between open surgical repair and EVAR recommendations?,NICE_NG156,Repairing unruptured aneurysms,40,NICE_NG156__p40-40__c0122,0.7949



10 clinical queries executed; all returned 5 chunks with metadata.
